In [1]:
import os
import math
import time
import sys
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
from tqdm import trange, tqdm
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

import torch
import torch.nn.functional as F  
from torch.utils.tensorboard import SummaryWriter
from problems.ThickenShell import ThickenShell
from Utils.CADTensorGenerator import CADTensorGenerator
from Utils.CADDomainVisualizer import CADDomainVisualizer
from Utils.CADVisualizer   import CADVisualizer
from neuraltomo_fem import run_fem_loss
from problems.ThickenShell import ThickenShell
from Training.MainTrain import NN_Trainer, TrainingConfig
from Decoder_CLasses.ContinuousVoronoiDecoder import ContinuousVoronoiDecoder
from Decoder_CLasses.NearUniformHoneycombBaseline import NearUniformHoneycombBaseline
from Decoder_CLasses.IntrinsicSurfaceHoneycombBaseline import IntrinsicSurfaceHoneycombBaseline
from Pred_NN_Classes.ppnet import PPNet
from scipy.spatial import Delaunay, Voronoi, voronoi_plot_2d, cKDTree,QhullError
from Utils.ExportAbaqus_VoxelBased import export_abaqus_voxel_fem
from Utils.ExportAbaqus import export_abaqus_shell
from Utils.embedded_centroid_stress_audit import (
    export_embedded_centroid_stress_audit,
    print_stress_case_summary,
)
from Utils.RunFEM_ExportVoxelABA import VoxelFEMAbaqusRunner

import pyvista as pv


# ---- Reproducibility (recommended for D_params comparisons) ----
SEED = 20
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

BASE = Path(__file__).parent if "__file__" in globals() else Path.cwd()
print("Code Directory:", BASE)
TesPartsDir = BASE / "Testparts" 
print("Test Step files Directory:", TesPartsDir)


if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device:", device)
# -------- PYVISTA BACKEND --------
def setup_pyvista(device):
    is_mac = sys.platform == "darwin"

    # Mac + MPS: prefer static to avoid VTK/trame hangs
    if is_mac :
        pv.OFF_SCREEN = True
        pv.set_jupyter_backend("static")
        backend = "static"
    else:
        try:
            pv.set_jupyter_backend("trame")
            backend = "trame"
        except Exception:
            pv.OFF_SCREEN = True
            pv.set_jupyter_backend("static")
            backend = "static"

    print(f"PyVista backend: {backend}")

setup_pyvista(device)

%matplotlib inline
def random_seeds_min_dist(N, min_dist=0.08, seed=1, max_tries=10000, device="cpu"):
    torch.manual_seed(seed)

    seeds = []
    tries = 0

    while len(seeds) < N and tries < max_tries:
        p = torch.rand(2, device=device)

        if len(seeds) == 0:
            seeds.append(p)
        else:
            current = torch.stack(seeds, dim=0)
            d = torch.linalg.norm(current - p[None, :], dim=-1)

            if d.min() >= min_dist:
                seeds.append(p)

        tries += 1

    if len(seeds) < N:
        raise RuntimeError(
            f"Could only generate {len(seeds)} seeds with min_dist={min_dist}."
        )

    return torch.stack(seeds, dim=0)

Code Directory: /home/arash/HVD_SeedsBase
Test Step files Directory: /home/arash/HVD_SeedsBase/Testparts
device: cuda
PyVista backend: trame


In [2]:

def to_numpy(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().numpy()
    return np.asarray(value)


def show_surface_density(fields, density_key="rho"):
    face_tensor = fields["face_tensor"]

    points = to_numpy(
        face_tensor["points_xyz"]
    ).astype(np.float64)

    triangles = to_numpy(
        face_tensor["faces_ijk"]
    ).astype(np.int64)

    # Convert one-based indexing if necessary.
    if triangles.min() == 1:
        triangles = triangles - 1

    # PyVista face format:
    # [3, i, j, k, 3, i, j, k, ...]
    pv_faces = np.column_stack(
        [
            np.full(triangles.shape[0], 3),
            triangles,
        ]
    ).reshape(-1)

    mesh = pv.PolyData(points, pv_faces)

    density = to_numpy(
        fields[density_key]
    ).reshape(-1)

    mesh.point_data["density"] = density

    plotter = pv.Plotter()
    plotter.add_mesh(
        mesh,
        scalars="density",
        cmap="viridis",
        clim=[0.0, 1.0],
        show_edges=False,
        scalar_bar_args={
            "title": density_key,
        },
    )

    plotter.add_text(
        f"Surface density: {density_key}",
        font_size=12,
    )

    plotter.show_axes()
    plotter.show()

In [ ]:
viz = CADVisualizer()
CircularSurf1  = TesPartsDir / "CircularSurf1.stp"
Planar = TesPartsDir / "Planar.stp"
FreeFormSurf3 = TesPartsDir / "FreeForm3.stp"
FreeSharp2 = TesPartsDir / "FreeSharp2.stp"
CircularHoles  = TesPartsDir / "CircularHoles.stp"
SphereTap      = TesPartsDir / "SphereTap.stp"
Helmet = TesPartsDir / "Helmet_New.stp"
FreeForm_Bend = TesPartsDir / "FreeForm_bend.stp"
CircBracket_Half = TesPartsDir / "CircBracket_Half.stp"

shape_path = FreeForm_Bend
Face_Cad = CADTensorGenerator(
    device=device,
    seed_domain_mask_res=512,
    mesh_size_scale = 0.2,
)

cad_domain, face_mesh = Face_Cad.generate_from_file(shape_path)
dx,dy,dz = Face_Cad.print_face_info();

points = face_mesh["points_xyz"].detach().cpu().numpy()
faces = face_mesh["pv_faces"].detach().cpu().numpy()

mesh = pv.PolyData(points, faces)

plotter = pv.Plotter()
plotter.add_mesh(
    mesh,
    show_edges=True,
)
plotter.add_axes()
plotter.show()

Info    : Clearing all models and views...
Info    : Done clearing all models and views
Info    :  - Label 'Shapes/Document' (2D)
Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (BSpline)
Info    : [ 30%] Meshing curve 2 (BSpline)
Info    : [ 60%] Meshing curve 3 (BSpline)
Info    : [ 80%] Meshing curve 4 (BSpline)
Info    : Done meshing 1D (Wall 0.0112084s, CPU 0.010935s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (BSpline surface, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 2.36215s, CPU 2.31947s)
Info    : 89426 nodes 178854 elements
Info    : Clearing all models and views...
Info    : Done clearing all models and views

=== Active Face Info ===
XYZ bounds:
  X: [-0.000000, 10.000000]  span=10.000000
  Y: [-2.500000, 2.500000]  span=5.000000
  Z: [-0.000000, 2.480848]  span=2.480848

UV bounds:
  U: [0.000000, 11.175616]
  V: [0.000000, 4.359416]

Periodic:
  U periodic: False
  V periodic: False



Widget(value='<iframe src="http://localhost:37691/index.html?ui=P_0x711878e51600_0&reconnect=auto" class="pyvi…

Exception raised
KeyError('acc1dee278f2a2aeb460a8b1def729c5_92967f')
Traceback (most recent call last):
  File "/home/arash/miniconda3/envs/VD_Imp_UB/lib/python3.10/site-packages/wslink/protocol.py", line 312, in onCompleteMessage
    results = func(*args, **kwargs)
  File "/home/arash/miniconda3/envs/VD_Imp_UB/lib/python3.10/site-packages/trame_vtk/modules/vtk/protocols/local_rendering.py", line 33, in get_array
    self.context.get_cached_data_array(data_hash, binary)
  File "/home/arash/miniconda3/envs/VD_Imp_UB/lib/python3.10/site-packages/trame_vtk/modules/vtk/serializers/synchronization_context.py", line 41, in get_cached_data_array
    cache_obj = self.data_array_cache[p_md5]
KeyError: 'acc1dee278f2a2aeb460a8b1def729c5_92967f'

Exception raised
KeyError('44a81fb3feb2fa015d874efdde733b53_154960L')
Traceback (most recent call last):
  File "/home/arash/miniconda3/envs/VD_Imp_UB/lib/python3.10/site-packages/wslink/protocol.py", line 312, in onCompleteMessage
    results = func(*arg

In [4]:
voxel_size = float((dx + dy + dz) / 250.0)
thickness = 3.0 * voxel_size
LoadingCase = "tensile_compression"
common_settings = dict(
    thickness=thickness,
    voxel_size=voxel_size,
    extra_layers=1,
    tensors=face_mesh,

    voxelization_mode="triangle_distance",
    subvoxel_samples=1,
    min_geom_fraction=0.25,

    bc_mapping_mode="surface_patch",
    faces_index_base=0,
)



In [5]:
print ("voxel_size:", voxel_size)

voxel_size: 0.06992339281788243


In [6]:
# shell_problem = ThickenShell(
#     **common_settings,

#     load_case="pin_bearing_bracket",
  

#     BC_dir="x",
#     fixed_side="max",

#     load_dir="x",
#     load_direction_side="min",
#     Load_magnitude=50.0,

#     bearing_loop_id=None,
#     bearing_contact_side="min",
#     bearing_pressure_exponent=1.0,
# )

In [7]:
shell_problem = ThickenShell(
    **common_settings,

    load_case="fixed_side_loading",

    # X identifies the two ends of the specimen.
    BC_dir="x",

    # Clamp the X-maximum end and load the X-minimum end.
    fixed_side="min",
    force_side="max",

    # Apply the force along the global Z-axis.
    load_dir="z",

    # "max" means positive Z.
    load_direction_side="max",

    # Total applied force = +5 N.
    Load_magnitude=0.000001,

    # Select only outward-facing voxel faces at the X-maximum end.
    fixed_region={
        "extent_axis": "x",
        "extent_side": "min",
        "band": 0.4 * voxel_size,
        "face_axis": "x",
        "face_side": "min",
    },

    # Select only outward-facing voxel faces at the X-minimum end.
    load_region={
        "extent_axis": "x",
        "extent_side": "max",
        "band": 0.4 * voxel_size,
        "face_axis": "x",
        "face_side": "max",
        "direction": "z",
        "total_force": 0.1,
    },

)

In [8]:
# shell_problem = ThickenShell(
#     **common_settings,

#     load_case="tensile_compression",
#     BC_dir="z",
#     Load_magnitude=-5.0,
# )
# loading_img = shell_problem.show_voxels_surface_and_bc(
#     return_img=True,
#     off_screen=True,
#     window_size=(560, 430),
#     show=True,
# )

In [9]:
loading_img = shell_problem.show_voxels_surface_and_bc(
    return_img=True,
    off_screen=True,
    window_size=(560, 430),
    show=True,
)

Widget(value='<iframe src="http://localhost:37691/index.html?ui=P_0x7118880fdf30_1&reconnect=auto" class="pyvi…

In [10]:
# shell_problem = ThickenShell(
#     **common_settings,

#     load_case="three_point_bending",

#     # Span direction
#     BC_dir="x",

#     # Load direction
#     load_dir="z",

#     # Downward force
#     Load_magnitude=-5.0,

#     # Restrict the middle load to the upper Z surface
#     load_surface_dir="z",
#     load_surface_side="max",
# )
# loading_img = shell_problem.show_voxels_surface_and_bc(
#     return_img=True,
#     off_screen=True,
#     window_size=(560, 430),
#     show=True,
# )


In [11]:
# shell_problem = ThickenShell(
#     **common_settings,

#     load_case="torsion",

#     # Torque axis and end-selection axis
#     BC_dir="x",

#     fixed_side="min",
#     force_side="max",

#     # Torque in N·mm when using N and mm
#     Load_magnitude=10.0,
# )

In [12]:
# shell_problem = ThickenShell(
#     **common_settings,

#     load_case="fixed_side_loading",

#     # Axis used to locate the two ends
#     BC_dir="x",

#     fixed_side="min",
#     force_side="max",

#     # Force direction
#     load_dir="z",

#     # "min" means negative Z
#     load_direction_side="min",

#     # Magnitude is converted to -10 N
#     Load_magnitude=5,
# )
# loading_img = shell_problem.show_voxels_surface_and_bc(
#     return_img=True,
#     off_screen=True,
#     window_size=(560, 430),
#     show=True,
# )

In [13]:
# voxel_size = float((dx + dy + dz) / 200.0)
# thickness = 3.0 * voxel_size

# shell_problem = ThickenShell(
#     thickness=thickness,
#     BC_dir="x",
#     Load_magnitude=2.0,
#     voxel_size=voxel_size,
#     extra_layers=1,
#     tensors=face_mesh,

#     voxelization_mode="triangle_distance",
#     subvoxel_samples=1,
#     min_geom_fraction=0.25,

#     bc_mapping_mode="surface_patch",
#     fixed_region={
#         "extent_axis": "x",
#         "extent_side": "min",
#         "band": 0.75 * voxel_size,
#     },
#     load_region={
#         "extent_axis": "x",
#         "extent_side": "max",
#         "band": 0.75 * voxel_size,
#         "direction": "x",
#         "total_force": 2.0,
#     },

#     load_case="tensile_compression",
# )

In [14]:
verification = VoxelFEMAbaqusRunner(
    face_mesh=face_mesh,
    shell_problem=shell_problem,
    device=device,
    output_dir="FEM_Verification",
)

/home/arash/HVD_SeedsBase/neuraltomo_fem/FE.py:31: RuntimeWarning: Legacy material keys Ef/Et/nuf/nut are mapped to explicit orthotropic CCF fields. Prefer material_E1/E2/E3, material_nu12/nu23/nu13, and material_G12/G23/G13.
  self.H8 = H8_anisotropic_K(device, element_size=self.mesh.elemSize, **problem.materialProperty)


In [15]:
Strut_Thickness = 0.2
decoder = ContinuousVoronoiDecoder(
    Cad_domain=Face_Cad,
    face_mesh=face_mesh,
    n_seeds=100,
    strut_thickness=Strut_Thickness,
    seed_domain_margin=0.6,
    # Keep your other settings unchanged.
)
baseline = NearUniformHoneycombBaseline(
    decoder=decoder,
    cad_domain=Face_Cad,
    face_mesh=face_mesh,

    target_seed_count=None,
    spacing_uv=0.15,

    mode="uv_hex",
    relaxation_steps=0,
    relaxation_factor=0.5,

    include_exterior_support_sites=True,
    support_band_cells=2.5,

    relax_support_sites=True,
    support_relaxation_neighbors=6,
    max_relaxation_displacement_cells=0.75,
    preserve_lattice_topology=True,
)

honeycomb_fields = baseline.generate(
    device=device,
    dtype=torch.float64,
    generate_density_fiber=True,
)

curves_uv = honeycomb_fields["edge_curves_uv"]          # [E, K, 2]
curves_xyz = honeycomb_fields["edge_curves_xyz"]        # [E, K, 3]
graph = honeycomb_fields["graph"]
honeycomb_fields["rho"] = ( honeycomb_fields["rho"] > 0.5).to(dtype=honeycomb_fields["rho"].dtype,device=honeycomb_fields["rho"].device,)
show_surface_density(honeycomb_fields, density_key="rho")





Widget(value='<iframe src="http://localhost:37691/index.html?ui=P_0x71158df49e10_2&reconnect=auto" class="pyvi…

In [16]:
honeycomb_fields["rho"] = ( honeycomb_fields["rho"] > 0.5).to(dtype=honeycomb_fields["rho"].dtype,device=honeycomb_fields["rho"].device,)
optimized_metrics = verification.run_fields(
    fields=honeycomb_fields,
    case_name="LL_Test" + shape_path.stem + "-Honeycomb-" + LoadingCase + f"_St{str(Strut_Thickness)[2:]}",
    material_bins=2,
    material_mass_density_kg_m3=1550.0, 
)
edge_lengths = honeycomb_fields["edge_curve_lengths_xyz"]          # [E]
total_all = honeycomb_fields["total_curve_length"]                 # scalar tensor
total_vd = honeycomb_fields["total_voronoi_curve_length"]          # scalar tensor


print("total all curve length:", float(total_all))
print("total VD curve length:", float(total_vd))

show_surface_density(honeycomb_fields, density_key="rho")


=== LL_TestFreeForm_bend-Honeycomb-tensile_compression_St2 ===
Maximum displacement magnitude: 0.95823597908 at [10.1039304733, -0.647030174732, -0.104885190725]
Compliance F^T U: 0.0939891040325
Maximum raw IP von Mises stress: 319.044128418 at [0.0201850440353, -0.312189787626, 0.0901084318757]
Abaqus input: /home/arash/HVD_SeedsBase/FEM_Verification/LL_TestFreeForm_bend-Honeycomb-tensile_compression_St2.inp
Summary JSON: /home/arash/HVD_SeedsBase/FEM_Verification/LL_TestFreeForm_bend-Honeycomb-tensile_compression_St2_summary.json
Embedded mass: 0.00798071929908 g; expected exported mass: 0.00798071929984 g
Mass difference abs/rel: 0 g / 0
total all curve length: 142.4049982400949
total VD curve length: 111.90139452120269


Widget(value='<iframe src="http://localhost:37691/index.html?ui=P_0x71158df4a830_3&reconnect=auto" class="pyvi…

In [17]:
from Training.MainTrain import (
    _field_to_numpy,
    load_optimized_shell_function,
    evaluate_optimized_shell_function,
    visualize_optimized_shell_fields,
)

Path1 = "/home/arash/HVD_SeedsBase/Case_Studies/FreeForm_bendFreeForm_bend_62_Sth2_tensile_compression1/optimized_shell_function.pt"
path2 = "/home/arash/HVD_SeedsBase/Case_Studies/PlanarPlannar_Seeds-70_Sth033/BestFeasibleCheckpoint/optimized_shell_function.pt"

Mpath =Path1
if not os.path.exists(Mpath):
    raise FileNotFoundError(f"Optimized shell function file not found: {Mpath}")
opt_fn = load_optimized_shell_function(Mpath, device="cuda")
fields = evaluate_optimized_shell_function(opt_fn)
fields["rho"] = (fields["rho"]>0.5).to(torch.float64)
optimized_metrics = verification.run_fields(
    fields=fields,
    case_name=shape_path.stem + "-OptimizedModel_Bend_St0174_F5_V80_St025",
    material_bins=2,
    material_mass_density_kg_m3=1550.0, 
)

total_all = fields["total_curve_length"]                 # scalar tensor
total_vd = fields["total_voronoi_curve_length"] 
print("total all curve length:", float(total_all))
print("total VD curve length:", float(total_vd))
show_surface_density(fields, density_key="rho")


=== FreeForm_bend-OptimizedModel_Bend_St0174_F5_V80_St025 ===
Maximum displacement magnitude: 0.656028151512 at [10.1039304733, 1.87021195889, -0.104885190725]
Compliance F^T U: 0.0638765543699
Maximum raw IP von Mises stress: 224.62171936 at [0.0201850440353, -0.801653504372, 0.0901084318757]
Abaqus input: /home/arash/HVD_SeedsBase/FEM_Verification/FreeForm_bend-OptimizedModel_Bend_St0174_F5_V80_St025.inp
Summary JSON: /home/arash/HVD_SeedsBase/FEM_Verification/FreeForm_bend-OptimizedModel_Bend_St0174_F5_V80_St025_summary.json
Embedded mass: 0.00847058516285 g; expected exported mass: 0.00847058516424 g
Mass difference abs/rel: 1.73472e-18 g / 2.04794e-16
total all curve length: 144.77944946289062
total VD curve length: 114.27584838867188


Widget(value='<iframe src="http://localhost:37691/index.html?ui=P_0x710e29d2b610_4&reconnect=auto" class="pyvi…

In [20]:
%load_ext tensorboard
%tensorboard --logdir runs

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 2270717), started 0:00:03 ago. (Use '!kill 2270717' to kill it.)